# DNA nanorulers (GATTA-PAINT 80Y and 80R) — no bias with the place on the Bayer pattern

**Makes up Figure:** the Bayer-phase SI figure (Reviewer 4, Major 2 b): `Nanoruler_BayerPhase.svg` (section 5).

**Question.** Does a molecule's measured position, a distance between two molecules, or its colour change with
where the molecule sits on the 2 × 2 colour-filter (Bayer) cell?

**No new experiment.** Each nanoruler acquisition drifts by several camera pixels over its 60,000 frames, so every
ruler is imaged at many places on the Bayer cell. The translation is the AIM drift estimate from the whole field,
not from the structure measured, so it cannot absorb a per-site bias.

**Method.**
1. The drift trace of each acquisition (AIM on the quality-cut localisations, exactly as `FigureSI_Nanorulers`).
2. The sites of every straight ruler (three sites, angle ≥ 170°, as `FigureSI_Nanorulers`); each linked event
   within 25 nm of a site belongs to it. Its raw camera position = drift-corrected position + drift at its frame,
   and its place in the Bayer cell = (raw position + 0.5) mod 2 px (pixel centres at integers).
3. Against that place, each referenced to the mean of all sites in the same 3,000-frame window (so a slow change
   over the acquisition, e.g. focus, or a drift-correction error common to the field, cannot pose as a Bayer
   effect: at any moment different rulers sit at different places on the pattern):
   - each ruler's two neighbour distances in each window, minus that ruler's mean distance;
   - each event's x and y offset from its site's centre;
   - each event's relative pixel 3 QE minus its site's mean.

**Data:** the fits and linked tables of `FigureSI_Nanorulers` (run it first, in this folder). With its
`FAST_MODE = True` (first TIFF of each acquisition, ~6,500 frames) the drift covers only part of the cell and the
maps are noisy; the published figure uses its full run.

---
## 0 — Imports

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# _nanorulers.py and _bayer_phase.py live one level up (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _nanorulers as nru
import _bayer_phase as bp

from pyS3M import IOFunctions
from pyS3M import SR_Functions

IO = IOFunctions.IO_Functions()
print("Python:", sys.version)

---
## 1 — Configuration

Folders, cuts, AIM segmentation and site finding as `FigureSI_Nanorulers`.

In [ ]:
PIXEL_NM = 69.0
RULERS = {
    "80Y": dict(folder="80Y_NanoRuler/20p561_DC1_LP561_BP582_64_75ms_TIRF_1", dye="Cy3B",
                cuts=dict(max_err_nm=40.0, sigma_nm=(80.0, 175.0),
                          min_colour={"A_B": 0.005, "A_G": 0.05, "A_R": 0.4}),
                aim_segmentation=10, sites=dict(eps_nm=10.0, min_events=50)),
    "80R": dict(folder="80R_NanoRuler/60mW_638_1_DC1_LP635_75ms_TIRF_1", dye="ATTO 655",
                cuts=dict(max_err_nm=40.0, sigma_nm=(100.0, 200.0),
                          min_colour={"A_B": 0.005, "A_G": 0.05, "A_R": 0.5}),
                aim_segmentation=20, sites=dict(eps_nm=10.0, min_events=200)),
}
LINK = dict(intersect_nm=20.0, roi_nm=60.0)
MIN_ANGLE_DEG = 170.0   # straight rulers only
ASSIGN_NM = 25.0        # an event belongs to a ruler site within this distance of its centre
WINDOW = 3000           # frames per time window (~0.2-0.4 px of drift)
MIN_WINDOW_EVENTS = 10  # events for a site's centre in one window
N_EXAMPLE = 6           # windows shown for the example rulers

SupRes_F = SR_Functions.SuperRes_Functions(camera="ximea", pixel_size=PIXEL_NM / 1000)

# ── Paths ──────────────────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().resolve().parents[2]
NR_ROOT   = Path("outputs") / "Nanorulers"   # FigureSI_Nanorulers' OUT_ROOT: its linked tables
NR_FITS   = NR_ROOT / "data"                 # its FIT_ROOT: each acquisition's Localisations.h5 + metadata
OUT_ROOT  = Path("outputs") / "FigureSI_Nanoruler_BayerPhase"   # git-ignored: drift traces, event tables
FIG_DIR   = OUT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Nanoruler fits  : {NR_FITS.resolve()}")
print(f"Linked tables   : {NR_ROOT.resolve()}")
print(f"Outputs         : {OUT_ROOT.resolve()}")

---
## 2 — Drift traces, ruler sites and their events

The linked tables of `FigureSI_Nanorulers` keep only drift-corrected positions, so the AIM trace is recomputed
once from the same quality-cut localisations (~5 min per acquisition; checked: corrected = raw − drift) and cached.
The colour filter of every pixel comes from the camera's mosaic at the acquisition's ROI origin.

In [ ]:
drift, events, rsites, units, n_frames = {}, {}, {}, {}, {}
for name, r in RULERS.items():
    folder = NR_FITS / r["folder"]
    meta = sorted(folder.glob("*metadata.txt"))[0]
    roi_x, roi_y, width, height = IO.metadata_reader_imageJ(str(meta))
    n_frames[name] = int(IO.metadata_nframes_reader_imageJ(str(meta)))
    masks = np.asarray(SupRes_F.mask.get_stacked_masks(roi_x, roi_y, width, height, SupRes_F.mosaic_unit))
    units[name] = np.array(list("BGR"))[np.argmax(masks[:2, :2], axis=-1)]   # filters of ROI rows/cols 0-1
    print(f"{name}: ROI origin column {roi_x}, row {roi_y}, {width} x {height} px; filters at the origin "
          f"{' / '.join(' '.join(row) for row in units[name])}")

    linked_file = NR_ROOT / f"{name}_linked.h5"
    record = dict(linked_bytes=linked_file.stat().st_size, fit_bytes=(folder / "Localisations.h5").stat().st_size,
                  cuts=r["cuts"], segmentation=r["aim_segmentation"], sites=r["sites"], min_angle=MIN_ANGLE_DEG,
                  assign_nm=ASSIGN_NM, window=WINDOW, **LINK)
    record = json.loads(json.dumps(record))
    rec_file = OUT_ROOT / f"{name}_record.json"
    files = {k: OUT_ROOT / f"{name}_{k}.pkl" for k in ("drift", "events", "sites")}
    if rec_file.exists() and json.loads(rec_file.read_text()) == record and all(f.exists() for f in files.values()):
        drift[name], events[name], rsites[name] = (pd.read_pickle(files[k]) for k in ("drift", "events", "sites"))
        print(f"{name}: reusing {OUT_ROOT}")
    else:
        cols = ["frame", "xc", "yc", "xc_err", "yc_err", "s_x", "s_y", "s_x_err", "s_y_err",
                "spot_matched_filter_response", "photoelectrons", "A_B", "A_G", "A_R"]
        kept, _ = nru.quality_cuts(pd.read_hdf(folder / "Localisations.h5", columns=cols), PIXEL_NM, **r["cuts"])
        drift[name], err = bp.drift_trace(kept, width, height, n_frames[name], PIXEL_NM, r["aim_segmentation"],
                                          LINK["intersect_nm"], LINK["roi_nm"])
        del kept
        print(f"{name}: corrected = raw - drift to {err:.1e} px")
        linked = pd.read_hdf(linked_file, columns=["frame", "xc", "yc", "xc_err", "yc_err", "A_R", "len"])
        sites = nru.find_sites(linked, PIXEL_NM, n_frames[name], **r["sites"])
        rsites[name] = bp.ruler_sites(sites, min_angle_deg=MIN_ANGLE_DEG)
        events[name] = bp.ruler_events(linked, rsites[name], drift[name], PIXEL_NM, ASSIGN_NM, WINDOW)
        del linked
        for k, v in (("drift", drift[name]), ("events", events[name]), ("sites", rsites[name])):
            v.to_pickle(files[k])
        rec_file.write_text(json.dumps(record, indent=1))
    d, e = drift[name], events[name]
    span = e.groupby("site").agg(rx=("raw_x", np.ptp), ry=("raw_y", np.ptp)).median()
    print(f"{name}: drift over {len(d):,} frames x {np.ptp(d.drift_x_px):.1f} px, y {np.ptp(d.drift_y_px):.1f} px; "
          f"{len(e):,} events on {e.site.nunique():,} sites of {rsites[name].ruler.nunique():,} straight rulers; "
          f"each site crossed a median {span.rx:.1f} x {span.ry:.1f} px of the camera\n")

---
## 3 — Offsets over the Bayer cell

Means in 0.5 px bins (4 × 4) of the 2 × 2 px cell, with the SEM. Printed: the largest |bin mean| of each map and the
median SEM of its bins (a bias would be a largest mean well above the SEM).

Example rulers (panels a, b): six 3,000-frame windows spread over the acquisition; the ruler whose sites have the
most events in its emptiest one.

In [ ]:
maps, examples = {}, {}
for name in RULERS:
    e, rs = events[name], rsites[name]
    piv, dd = bp.window_distances(e, PIXEL_NM, MIN_WINDOW_EVENTS)
    maps[name] = dict(dist=bp.cell_map(dd.dd, dd.u, dd.v), dx=bp.cell_map(e.dx_w, e.u, e.v),
                      dy=bp.cell_map(e.dy_w, e.u, e.v), col=bp.cell_map(e.dAR_w, e.u, e.v))
    print(f"{name}: {len(piv):,} ruler-windows from {piv.index.get_level_values(0).nunique():,} rulers; "
          f"distance about each ruler's own mean: SD {dd.dd.std():.2f} nm")
    for key, what, unit, sc in (("dist", "ruler distance", "nm", 1), ("dx", "x offset", "nm", 1),
                                ("dy", "y offset", "nm", 1), ("col", "rel. pixel 3 QE", "x 1e-3", 1e3)):
        m, s = maps[name][key]
        print(f"   {what:16s} ({unit}): largest |mean| {np.nanmax(np.abs(m)) * sc:.2f}, "
              f"median SEM {np.nanmedian(s) * sc:.2f}")
    n_win = int(e.win.max()) + 1
    wins = np.unique(np.round(np.linspace(0, n_win - 1, N_EXAMPLE)).astype(int))
    counts = piv["n"].min(axis=1).unstack("win").reindex(columns=wins).fillna(0)   # 0: a site missing
    best = counts.min(axis=1).idxmax()
    examples[name] = dict(ruler=best, wins=wins, events=e[(e.ruler == best) & e.win.isin(wins)].copy(),
                          sites=rs[rs.ruler == best].sort_values("pos"))
    print(f"   example ruler {best}: fewest events per site in a shown window {counts.loc[best].min():.0f}\n")

---
## 4 — Figure

a) one ruler's drift across the camera (smoothed over 200 frames), over each pixel's colour filter; the windows of
b) in colour. b) that ruler's events in each window at their raw camera positions, over the filters they fell on
(crosses: the window's site centres; below: its two neighbour distances). c) ruler neighbour distance in each window
minus the ruler's mean; d), e) the mean x and y offsets of events from their site centre; f) relative pixel 3 QE minus
the site mean; each over the place in the 2 × 2 Bayer cell (outlined in its filters' colours), referenced to all
sites in the same window.

In [ ]:
PALETTE = bp.iwanthue(N_EXAMPLE, chroma=(40, 90), lightness=(30, 62), seed=3)
SPECS = [("dist", "ruler distance $-$ its mean (nm)", 1.0, 0.5, "c"),     # key, label, scale, colour limit, panel
         ("dx", "x offset from site centre (nm)", 1.0, 0.25, "d"),
         ("dy", "y offset from site centre (nm)", 1.0, 0.25, "e"),
         ("col", r"rel. pixel 3 QE $-$ mean ($\times10^{-3}$)", 1e3, 2.0, "f")]
HALF = 3.0   # b): px either side of the ruler centre

MM = 1 / 25.4; Wf, Hf = 170.0, 184.0
fig = plt.figure(figsize=(Wf * MM, Hf * MM)); fig.set_layout_engine("none")
def add(x, y, w, h): return fig.add_axes([x / Wf, 1 - (y + h) / Hf, w / Wf, h / Hf])
def label(ax, s, dx=-0.18, dy=1.03):
    ax.text(dx, dy, f"{s})", transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom")

# a) the path of the example ruler's middle site across the sensor, over the colour filters
for j, name in enumerate(RULERS):
    ex, unit = examples[name], units[name]
    sx, sy = ex["sites"].x.to_numpy()[1] / PIXEL_NM, ex["sites"].y.to_numpy()[1] / PIXEL_NM
    px = pd.Series(sx + drift[name].drift_x_px.to_numpy()).rolling(200, center=True, min_periods=1).mean().to_numpy()
    py = pd.Series(sy + drift[name].drift_y_px.to_numpy()).rolling(200, center=True, min_periods=1).mean().to_numpy()
    span = max(np.ptp(px), np.ptp(py)) + 1.6
    cx, cy = (px.min() + px.max()) / 2, (py.min() + py.max()) / 2
    ax = add(16 + j * 80, 6, 52, 52)
    bp.draw_mosaic(ax, unit, cx - span / 2, cx + span / 2, cy - span / 2, cy + span / 2)
    ax.plot(px, py, color="0.35", lw=0.6, zorder=2)
    for k, w in enumerate(ex["wins"]):
        sl = slice(w * WINDOW, (w + 1) * WINDOW)
        ax.plot(px[sl], py[sl], color=PALETTE[k], lw=2.4, zorder=3, solid_capstyle="round")
        ax.text(px[sl].mean(), py[sl].mean(), str(k + 1), fontsize=6, ha="center", va="center", color="white",
                fontweight="bold", zorder=4)
    ax.set_xlim(cx - span / 2, cx + span / 2); ax.set_ylim(cy + span / 2, cy - span / 2); ax.set_aspect("equal")
    ax.set_xlabel("camera column (px)", fontsize=7)
    if j == 0: ax.set_ylabel("camera row (px)", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.text(0.5, 1.02, f"{name} ({RULERS[name]['dye']}): one ruler's drift", transform=ax.transAxes,
            ha="center", fontsize=7)
    if j == 0: label(ax, "a", dx=-0.3)
for k, f in enumerate("BGR"):   # filter legend
    lax = add(146, 14 + k * 7, 4, 4)
    lax.add_patch(Rectangle((0, 0), 1, 1, facecolor=bp.TINT[f], alpha=0.5, edgecolor="none"))
    lax.set_xlim(0, 1); lax.set_ylim(0, 1); lax.axis("off")
    lax.text(1.4, 0.5, f"pixel {k + 1} filter", fontsize=6, va="center", transform=lax.transAxes)

# b) the ruler in each window, at its raw camera position, over the filters it sits on
for j, name in enumerate(RULERS):
    ex, unit = examples[name], units[name]
    ev = ex["events"]
    for k, w in enumerate(ex["wins"]):
        sub = ev[ev.win == w]
        cxp, cyp = sub.raw_x.mean(), sub.raw_y.mean()
        ax = add(14 + k * 26.2, 70 + j * 28, 23, 23)
        bp.draw_mosaic(ax, unit, cxp - HALF, cxp + HALF, cyp - HALF, cyp + HALF, scale=PIXEL_NM, alpha=0.32, lw=0.6)
        ax.scatter(sub.raw_x * PIXEL_NM, sub.raw_y * PIXEL_NM, s=0.6, color=PALETTE[k], lw=0, zorder=2)
        cen = sub.groupby("pos")[["raw_x", "raw_y"]].mean() * PIXEL_NM
        ax.scatter(cen.raw_x, cen.raw_y, marker="+", s=14, color="black", lw=0.6, zorder=3)
        d = np.hypot(np.diff(cen.raw_x), np.diff(cen.raw_y))
        ax.set_xlim((cxp - HALF) * PIXEL_NM, (cxp + HALF) * PIXEL_NM)
        ax.set_ylim((cyp + HALF) * PIXEL_NM, (cyp - HALF) * PIXEL_NM)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_color(PALETTE[k]); s.set_linewidth(1.4)
        ax.text(0.04, 0.96, str(k + 1), transform=ax.transAxes, fontsize=6.5, fontweight="bold", va="top",
                color=PALETTE[k])
        ax.text(0.5, -0.03, ", ".join(f"{x:.0f}" for x in d) + " nm", transform=ax.transAxes, fontsize=5.5,
                ha="center", va="top")
        if k == 0:
            ax.text(-0.12, 0.5, name, transform=ax.transAxes, fontsize=7, rotation=90, va="center", ha="center")
            if j == 0: label(ax, "b", dx=-0.3, dy=1.04)
            x0b, y0b = (cxp - HALF + 0.3) * PIXEL_NM, (cyp + HALF - 0.4) * PIXEL_NM
            ax.plot([x0b, x0b + 100], [y0b] * 2, color="black", lw=1.2)
            ax.text(x0b + 50, y0b - 0.15 * PIXEL_NM, "100 nm", fontsize=5, ha="center", va="bottom")

# c-f) the measured offsets over the Bayer cell
for k, (key, lab, sc, lim, letter) in enumerate(SPECS):
    x0 = 14 + k * 40.5
    for j, name in enumerate(RULERS):
        ax = add(x0, 133 + j * 25, 22, 22)
        im = ax.imshow(maps[name][key][0] * sc, extent=[0, 2, 2, 0], cmap="RdBu_r", vmin=-lim, vmax=lim, zorder=1)
        bp.cell_axes(ax, units[name])
        ax.text(-0.12, 0.5, name, transform=ax.transAxes, fontsize=6.5, rotation=90, va="center", ha="center")
        if j == 0:
            label(ax, letter, dx=-0.32)
            ax.text(0.5, 1.04, "position in the Bayer cell", transform=ax.transAxes, fontsize=6, ha="center",
                    va="bottom")
    cb = fig.colorbar(im, cax=add(x0 + 23.5, 133, 1.3, 47))
    cb.ax.tick_params(labelsize=5.5)
    cb.set_label(lab, fontsize=6)
fig.savefig(FIG_DIR / "Nanoruler_BayerPhase.svg", dpi=600, format="svg")
print(f"Saved {FIG_DIR / 'Nanoruler_BayerPhase.svg'}")
plt.show()